In [0]:
print("Initializing Gold Layer Aggregations...")

# 1. Reading the clean Silver data (Parquet)
silver_path = "/Volumes/project_streame/silver_layer/cleaned_data/"
clean_df = spark.read.parquet(silver_path)

# 2. The Bridge to SQL is Creating a Temporary View
# This allows us to query the DataFrame exactly like a standard database table
clean_df.createOrReplaceTempView("ecommerce_events")

# 3. Writing SQL to build our first Gold Table: Device Analytics
# How many users are performing which actions on which devices?
device_metrics_df = spark.sql("""
    SELECT 
        device,
        action,
        COUNT(event_id) as total_events,
        COUNT(DISTINCT user_id) as unique_users
    FROM ecommerce_events
    GROUP BY device, action
    ORDER BY device, total_events DESC
""")

print("Gold Table 1: Device Activity Metrics")
display(device_metrics_df)

# 4. Writing a SQL to build our second Gold Table: Revenue Analytics
# What is the total revenue generated per item?
revenue_metrics_df = spark.sql("""
    SELECT 
        item_id,
        COUNT(event_id) as total_purchases,
        SUM(price) as total_revenue_generated
    FROM ecommerce_events
    WHERE action = 'purchase'
    GROUP BY item_id
    ORDER BY total_revenue_generated DESC
""")

print("\nGold Table 2: Top Performing Items")
display(revenue_metrics_df)

# 5. Save the Gold Tables to Unity Catalog
gold_path_devices = "/Volumes/project_streame/gold_layer/device_metrics/"
gold_path_revenue = "/Volumes/project_streame/gold_layer/revenue_metrics/"

device_metrics_df.write.format("parquet").mode("overwrite").save(gold_path_devices)
revenue_metrics_df.write.format("parquet").mode("overwrite").save(gold_path_revenue)

print(f"\nSuccess! Gold metrics saved permanently to {gold_path_devices} and {gold_path_revenue}")